# 初期セットアップ（ステップ３）
このノートブックでは、サンプルコードの実行環境を初めて使用する際の設定のステップ３を行います。  
ステップ３では、準備されたドメインの管理権を確認し、configファイルにドメインIDを書き込みます。  

## 設定項目（実行前に、書き換えてください）

事前に作成したドメインのIDを、下のセルを編集して指定してください。  
ドメインの作成については、[こちら](https://github.com/dncware-blockchain-plus/samples-notebook/blob/main/DOMAIN.md)を参照してください。

In [1]:
var domainID = "d93319890"; // 自分のドメインIDに書き換えてください

## 必要なライブラリの読み込み

In [2]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { adminWallet, rpc } = await import('../lib/notebook-util.mjs');

## 動作確認

指定されたドメインが存在するか確認します。  
エラーになる場合は、事前の作成が間違えている可能性があります。

In [3]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'a_domain', id: domainID });
if (resp.status !== 'ok') {
    console.log(resp);
    console.error('ドメインが見つかりません');
    var success_1 = false;
} else {
    console.log('OK');
    var success_1 = true;
}

OK


ステップ１で作成したウォレットがブロックチェーンにユーザ登録されているか確認します。  
エラーになる場合は、事前の作成が間違えている可能性があります。

In [4]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'search', key: 'me' });
var uid = resp.value[0].id;
if (uid === 'anonymous') {
    console.error('ユーザが登録されていません');
    var success_2 = false;
} else {
    console.log('OK');
    var success_2 = true;
}

OK


上記ユーザがしてされたドメインの管理権限を有しているか確認します。  
(削除権限を付与する管理操作が成功することを確認します）  
エラーになる場合は、事前の作成が間違えている可能性があります。

In [5]:
var resp = await rpc.call(adminWallet, 'c1update', { id: domainID, prop: 'add terminated_by', value: [uid] });
if (resp.status !== 'ok') {
    console.log(resp);
    console.error('管理権限がありません');
    var success_3 = false;
} else {
    console.log('OK');
    var success_3 = true;
}

OK


## 設定ファイルへの反映
設定ファイル(etc/config.json)へドメインの設定を書き込みます。

In [6]:
if(success_1 !== true || success_2 !== true || success_3 !== true) throw '確認が成功していません';
var fs = await import('node:fs');
var path = await import('node:path');
var { default: package_root } = await import('../lib/get-package-root.mjs');
var config_filename = path.join(package_root, 'etc', 'config.json');
if(!fs.existsSync(config_filename)) throw '設定ファイルが存在しません:'+config_filename;
var config = JSON.parse(fs.readFileSync(config_filename, 'utf8'));
config.domain = domainID;
fs.writeFileSync(config_filename, JSON.stringify(config, null, 1), 'utf8');
console.log('DONE');

DONE
